# PHASE 2 KOMPONEN 2 — LANGKAH 1: KONSTRUKSI NETWORK GRAPH
## Social Penetration & Network Diffusion Framework

**Input:** `Data_bert_with_topics.csv` (29.289 tweets dengan topic labels dari BERTopic)

**Output:**
- `network_edges.csv` — weighted edge list
- `network_nodes.csv` — node list dengan atribut

**Spesifikasi graf:**
- Directed graph
- Weighted edges (multiple interaksi = weight lebih tinggi)
- 3 tipe edge: reply, quote, mention

## Cell 1: Install & Import

In [19]:
!pip install networkx python-louvain -q

import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter

print("All libraries imported successfully.")

All libraries imported successfully.


## Cell 2: Load Data

In [20]:
df = pd.read_csv('Data_bert_with_topics.csv')

print(f"Total tweets: {len(df)}")
print(f"Unique users: {df['username'].nunique()}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAccount type distribution:")
print(df['account_type'].value_counts())

Total tweets: 29289
Unique users: 19621
Columns: ['date', 'is_quote', 'mentions', 'username', 'verified', 'account_type', 'full_text', 'tweet_url', 'quoted_url', 'view_count', 'quote_count', 'quoted_text', 'reply_count', 'display_name', 'retweet_count', 'favorite_count', 'in_reply_to_url', 'quoted_username', 'user_statuses_count', 'user_followers_count', 'user_following_count', 'in_reply_to_screen_name', 'topic', 'probability', 'topic_label']

Account type distribution:
account_type
grassroot          28505
media/news           659
elite/political      125
Name: count, dtype: int64


## Cell 3: Cleaning — Strip @ dari semua kolom username

Kolom `mentions` bisa berisi multiple mentions (`@userA @userB @userC`), jadi perlu di-split.
Semua username di-lowercase untuk konsistensi matching.

In [21]:
def clean_username(val):
    """Strip @ dan lowercase untuk konsistensi matching."""
    if pd.isna(val):
        return np.nan
    return str(val).strip().lstrip('@').lower()

def clean_mentions(val):
    """Split multiple mentions, strip @, return list."""
    if pd.isna(val):
        return []
    mentions = str(val).strip().split()
    return [m.lstrip('@').lower() for m in mentions if m.lstrip('@').lower()]

df['username_clean'] = df['username'].apply(clean_username)
df['reply_to_clean'] = df['in_reply_to_screen_name'].apply(clean_username)
df['quoted_clean'] = df['quoted_username'].apply(clean_username)
df['mentions_clean'] = df['mentions'].apply(clean_mentions)

# Verifikasi
print("Sample username_clean:")
print(df['username_clean'].head(3).tolist())
print(f"\nSample reply_to_clean (non-null):")
print(df['reply_to_clean'].dropna().head(3).tolist())
print(f"\nSample mentions_clean (non-empty):")
print(df['mentions_clean'][df['mentions_clean'].str.len() > 0].head(3).tolist())

# Sanity check: berapa tweet yang punya masing-masing tipe relasi?
print(f"\n--- Relasi tersedia ---")
print(f"Tweets with reply: {df['reply_to_clean'].notna().sum()}")
print(f"Tweets with quote: {df['quoted_clean'].notna().sum()}")
print(f"Tweets with mention: {(df['mentions_clean'].str.len() > 0).sum()}")

Sample username_clean:
['solehudien30', 'k0dhar', 'arianadezhu']

Sample reply_to_clean (non-null):
['goodrecom', 'bebekmerayap', 'itschicks']

Sample mentions_clean (non-empty):
[['goodrecom'], ['bebekmerayap'], ['itschicks', 'awesomeposted']]

--- Relasi tersedia ---
Tweets with reply: 6253
Tweets with quote: 2969
Tweets with mention: 8056


## Cell 4: Build Edge List

Tiga sumber edge:
1. **Reply**: `username_clean` → `reply_to_clean`
2. **Quote**: `username_clean` → `quoted_clean`
3. **Mention**: `username_clean` → setiap user di `mentions_clean`

Self-loop difilter. Setiap edge membawa metadata: tipe, timestamp, dan topic.

In [22]:
edges_raw = []

for idx, row in df.iterrows():
    source = row['username_clean']
    timestamp = row['date']
    topic = row['topic']

    # Edge tipe 1: Reply
    if pd.notna(row['reply_to_clean']):
        target = row['reply_to_clean']
        if target != source:  # filter self-loop
            edges_raw.append({
                'source': source,
                'target': target,
                'type': 'reply',
                'date': timestamp,
                'topic': topic
            })

    # Edge tipe 2: Quote
    if pd.notna(row['quoted_clean']):
        target = row['quoted_clean']
        if target != source:  # filter self-loop
            edges_raw.append({
                'source': source,
                'target': target,
                'type': 'quote',
                'date': timestamp,
                'topic': topic
            })

    # Edge tipe 3: Mention (bisa multiple per tweet)
    for mentioned_user in row['mentions_clean']:
        if mentioned_user != source:  # filter self-loop
            edges_raw.append({
                'source': source,
                'target': mentioned_user,
                'type': 'mention',
                'date': timestamp,
                'topic': topic
            })

edges_df = pd.DataFrame(edges_raw)

print(f"Total raw edges: {len(edges_df)}")
print(f"\nEdge type distribution:")
print(edges_df['type'].value_counts())
print(f"\nUnique source nodes: {edges_df['source'].nunique()}")
print(f"Unique target nodes: {edges_df['target'].nunique()}")

Total raw edges: 21420

Edge type distribution:
type
mention    13133
reply       5473
quote       2814
Name: count, dtype: int64

Unique source nodes: 7173
Unique target nodes: 4726


## Cell 5: Aggregate Weighted Edges

Jika user A berinteraksi dengan user B sebanyak 3 kali (misal: 1 reply + 2 mention),
maka dijadikan 1 edge dengan weight = 3.

Metadata yang disimpan per edge:
- `weight`: jumlah total interaksi
- `dominant_type`: tipe interaksi paling sering (reply/quote/mention)
- `dominant_topic`: topik paling sering dibahas dalam interaksi tersebut
- `first_interaction` dan `last_interaction`: rentang waktu

In [23]:
weighted_edges = edges_df.groupby(['source', 'target']).agg(
    weight=('type', 'size'),
    edge_types=('type', lambda x: list(x)),
    topics=('topic', lambda x: list(x)),
    first_interaction=('date', 'min'),
    last_interaction=('date', 'max')
).reset_index()

# Dominant edge type per pair
weighted_edges['dominant_type'] = weighted_edges['edge_types'].apply(
    lambda x: Counter(x).most_common(1)[0][0]
)

# Dominant topic per pair
weighted_edges['dominant_topic'] = weighted_edges['topics'].apply(
    lambda x: Counter(x).most_common(1)[0][0]
)

print(f"Total unique weighted edges: {len(weighted_edges)}")
print(f"\nWeight distribution:")
print(weighted_edges['weight'].describe())
print(f"\nEdges with weight > 1: {(weighted_edges['weight'] > 1).sum()}")
print(f"Edges with weight > 5: {(weighted_edges['weight'] > 5).sum()}")
print(f"Max weight: {weighted_edges['weight'].max()}")

print(f"\nDominant type distribution:")
print(weighted_edges['dominant_type'].value_counts())

Total unique weighted edges: 13994

Weight distribution:
count    13994.000000
mean         1.530656
std          1.518869
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         96.000000
Name: weight, dtype: float64

Edges with weight > 1: 5461
Edges with weight > 5: 130
Max weight: 96

Dominant type distribution:
dominant_type
mention    6497
reply      4904
quote      2593
Name: count, dtype: int64


## Cell 6: Build Directed Graph

In [24]:
G = nx.DiGraph()

# Add edges with attributes
for _, row in weighted_edges.iterrows():
    G.add_edge(
        row['source'],
        row['target'],
        weight=row['weight'],
        dominant_type=row['dominant_type'],
        dominant_topic=row['dominant_topic'],
        first_interaction=row['first_interaction'],
        last_interaction=row['last_interaction']
    )

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")

Nodes: 11426
Edges: 13994
Density: 0.000107


## Cell 7: Assign Node Attributes (account_type)

Node yang hanya muncul sebagai target (di-mention/di-reply/di-quote)
tapi tidak pernah posting tweet sendiri di dataset akan dapat label `unknown`.

In [25]:
# Buat mapping username -> account_type dari dataset
user_type_map = df.groupby('username_clean')['account_type'].first().to_dict()

# Assign ke node
for node in G.nodes():
    G.nodes[node]['account_type'] = user_type_map.get(node, 'unknown')

# Cek distribusi
type_counts = Counter(nx.get_node_attributes(G, 'account_type').values())
print("Node account_type distribution:")
for t, c in type_counts.most_common():
    pct = c / G.number_of_nodes() * 100
    print(f"  {t}: {c} ({pct:.1f}%)")

print(f"\nTotal nodes in graph: {G.number_of_nodes()}")
print(f"Nodes from dataset: {len(user_type_map)}")
print(f"Nodes only as targets (unknown): {type_counts.get('unknown', 0)}")

Node account_type distribution:
  grassroot: 7506 (65.7%)
  unknown: 3840 (33.6%)
  media/news: 48 (0.4%)
  elite/political: 32 (0.3%)

Total nodes in graph: 11426
Nodes from dataset: 19621
Nodes only as targets (unknown): 3840


## Cell 8: Basic Network Statistics

In [26]:
print("=" * 50)
print("NETWORK SUMMARY")
print("=" * 50)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.6f}")
print(f"Weakly connected components: {nx.number_weakly_connected_components(G)}")
print(f"Strongly connected components: {nx.number_strongly_connected_components(G)}")

# Largest weakly connected component
largest_wcc = max(nx.weakly_connected_components(G), key=len)
print(f"Largest WCC size: {len(largest_wcc)} ({len(largest_wcc)/G.number_of_nodes()*100:.1f}% of nodes)")

# Degree stats
in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]
print(f"\nIn-degree  — mean: {np.mean(in_degrees):.2f}, median: {np.median(in_degrees):.0f}, max: {max(in_degrees)}")
print(f"Out-degree — mean: {np.mean(out_degrees):.2f}, median: {np.median(out_degrees):.0f}, max: {max(out_degrees)}")

# Top 5 nodes by in-degree (attention sinks)
top_in = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:5]
print(f"\nTop 5 in-degree (attention sinks):")
for user, deg in top_in:
    print(f"  {user}: {deg} (type: {G.nodes[user].get('account_type', 'unknown')})")

# Top 5 nodes by out-degree (broadcasters/diffusion drivers)
top_out = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:5]
print(f"\nTop 5 out-degree (broadcasters):")
for user, deg in top_out:
    print(f"  {user}: {deg} (type: {G.nodes[user].get('account_type', 'unknown')})")

NETWORK SUMMARY
Nodes: 11426
Edges: 13994
Density: 0.000107
Weakly connected components: 1616
Strongly connected components: 11353
Largest WCC size: 7137 (62.5% of nodes)

In-degree  — mean: 1.22, median: 0, max: 621
Out-degree — mean: 1.22, median: 1, max: 84

Top 5 in-degree (attention sinks):
  prabowo: 621 (type: unknown)
  dpr_ri: 483 (type: unknown)
  txtdrimedia: 307 (type: unknown)
  barengwarga: 261 (type: elite/political)
  tempodotco: 206 (type: grassroot)

Top 5 out-degree (broadcasters):
  adhi_faiz: 84 (type: grassroot)
  bincjun: 78 (type: grassroot)
  jannahseeker99: 70 (type: grassroot)
  van88xier: 65 (type: grassroot)
  rahmaniarbaftim: 62 (type: grassroot)


## Cell 9: Save Outputs

Dua file output:
- `network_edges.csv` — weighted edge list untuk Langkah 2 (Community Detection)
- `network_nodes.csv` — node list dengan atribut untuk analisis aktor

In [27]:
# Save edge list (drop list columns yang tidak bisa di-Excel dengan bersih)
edges_save = weighted_edges[['source', 'target', 'weight', 'dominant_type',
                              'dominant_topic', 'first_interaction', 'last_interaction']].copy()
edges_save.to_csv('network_edges.csv', index=False)

# Save node list with attributes
node_data = []
for node in G.nodes():
    node_data.append({
        'username': node,
        'account_type': G.nodes[node].get('account_type', 'unknown'),
        'in_degree': G.in_degree(node),
        'out_degree': G.out_degree(node),
        'total_degree': G.in_degree(node) + G.out_degree(node)
    })
node_df = pd.DataFrame(node_data)
node_df.to_csv('network_nodes.csv', index=False)

print(f"Saved: network_edges.csv ({len(edges_save)} edges)")
print(f"Saved: network_nodes.csv ({len(node_df)} nodes)")
print(f"\n--- Langkah 1 selesai. Ready untuk Langkah 2 (Community Detection). ---")

Saved: network_edges.csv (13994 edges)
Saved: network_nodes.csv (11426 nodes)

--- Langkah 1 selesai. Ready untuk Langkah 2 (Community Detection). ---
